# Практика №1. Линейная регрессия

**Задача:** предсказать прочность бетона на сжатие (МПа) по составу смеси и возрасту образца.

**Датасет:** UCI Concrete Compressive Strength, 1 030 наблюдений, 8 числовых признаков и непрерывная целевая переменная. Источник: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/165/concrete-compressive-strength), DOI: `10.24432/C5PK67`.

В работе последовательно выполняются загрузка данных, проверка качества, разведочный анализ, разделение на обучающую/валидационную/тестовую выборки, обучение и итоговая оценка модели.

In [ ]:
%pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 1. Загрузка данных

Используется `ucimlrepo`, который загружает набор непосредственно из UCI. Целевая переменная берётся отдельно от признаков, а затем объединяется с ними только для проверки качества строк.

In [ ]:
dataset = fetch_ucirepo(id=165)
X_raw = dataset.data.features.copy()
y_raw = dataset.data.targets.iloc[:, 0].copy()

print(f"Размер матрицы признаков: {X_raw.shape}")
print(f"Целевая переменная: {dataset.data.targets.columns[0]}")
display(X_raw.head())
display(y_raw.describe().to_frame(name="compressive_strength_mpa"))

## 2. Проверка качества и очистка

Проверяем пропуски, бесконечные значения, полные дубликаты и статистические выбросы по правилу межквартильного размаха (IQR). Выбросы не удаляются автоматически: экстремальная прочность может быть корректным результатом конкретного состава или возраста бетона. В модели пропуски признаков будут заполняться медианой, рассчитанной только по обучающей выборке.

In [ ]:
df = X_raw.copy()
df["target"] = pd.to_numeric(y_raw, errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)

print("Пропуски до очистки:")
display(df.isna().sum().to_frame("missing"))
print("Количество полных дубликатов:", int(df.duplicated().sum()))

rows_before = len(df)
df = df.dropna(subset=["target"]).drop_duplicates().reset_index(drop=True)
print(f"Удалено строк с отсутствующей целевой переменной/дубликатов: {rows_before - len(df)}")
print("Пропуски в признаках после проверки:")
display(df.drop(columns="target").isna().sum().to_frame("missing"))

X = df.drop(columns="target")
y = df["target"]

def count_iqr_outliers(frame):
    counts = {}
    for column in frame.select_dtypes(include=np.number).columns:
        q1, q3 = frame[column].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        counts[column] = int(((frame[column] < lower) | (frame[column] > upper)).sum())
    return pd.Series(counts, name="iqr_outliers").sort_values(ascending=False)

print("Потенциальные выбросы по IQR (не удаляются автоматически):")
display(count_iqr_outliers(X).to_frame())
display(df.describe().T)

## 3. Разведочный анализ данных (EDA)

Построим распределения всех признаков и целевой переменной. Матрица корреляции помогает увидеть линейные связи между числовыми переменными, но сама по себе не доказывает причинно-следственные отношения.

In [ ]:
plot_columns = list(X.columns) + ["target"]
df[plot_columns].hist(figsize=(15, 11), bins=25, edgecolor="black")
plt.suptitle("Распределения признаков и прочности бетона", y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица")
plt.tight_layout()
plt.show()

## 4. Разделение выборки и подготовка признаков

Сначала отделяется тестовая часть, которая не используется при выборе модели. Оставшиеся данные делятся на train и validation. Итого получаем приблизительно 70% / 15% / 15%. Импутация и масштабирование находятся внутри `Pipeline`, поэтому их параметры рассчитываются только на обучающих данных и не приводят к утечке информации.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} строк")
print(f"Validation: {X_val.shape[0]} строк")
print(f"Test: {X_test.shape[0]} строк")

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

## 5. Обучение линейной регрессии

Линейная регрессия подбирает коэффициенты линейной зависимости между входными признаками и целевой переменной. Сначала оцениваем модель на validation, затем обучаем окончательную модель на объединении train и validation.

In [ ]:
validation_model = Pipeline([
    ("preprocessor", numeric_preprocessor),
    ("model", LinearRegression()),
])
validation_model.fit(X_train, y_train)
y_val_pred = validation_model.predict(X_val)

validation_metrics = {
    "MAE": mean_absolute_error(y_val, y_val_pred),
    "RMSE": np.sqrt(mean_squared_error(y_val, y_val_pred)),
    "R2": r2_score(y_val, y_val_pred),
}
print("Метрики на validation:")
display(pd.Series(validation_metrics).to_frame("value"))

X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

final_model = Pipeline([
    ("preprocessor", numeric_preprocessor),
    ("model", LinearRegression()),
])
final_model.fit(X_train_full, y_train_full)
y_test_pred = final_model.predict(X_test)

test_metrics = {
    "MAE": mean_absolute_error(y_test, y_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
    "R2": r2_score(y_test, y_test_pred),
}
print("Итоговые метрики на отложенной test-выборке:")
display(pd.Series(test_metrics).to_frame("value"))

## 6. Анализ ошибок и сохранение модели

MAE показывает среднюю абсолютную ошибку в МПа, RMSE сильнее штрафует крупные ошибки, а R² отражает долю объяснённой вариации целевой переменной. Проверим распределение остатков и сохраним обученный Pipeline вместе с метриками.

In [ ]:
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(x=y_test, y=y_test_pred, ax=axes[0], alpha=0.7)
min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_value, max_value], [min_value, max_value], "--", color="black")
axes[0].set_title("Факт vs прогноз")
axes[0].set_xlabel("Фактическая прочность, МПа")
axes[0].set_ylabel("Предсказанная прочность, МПа")

axes[1].hist(residuals, bins=25, density=True, alpha=0.75)
axes[1].set_title("Распределение остатков")
axes[1].set_xlabel("Остаток, МПа")
plt.tight_layout()
plt.show()

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, artifact_dir / "linear_regression_concrete.joblib")
with (artifact_dir / "linear_regression_metrics.json").open("w", encoding="utf-8") as file:
    json.dump({"validation": validation_metrics, "test": test_metrics}, file, ensure_ascii=False, indent=2)

print("Модель и метрики сохранены в папку artifacts/")

## 7. Вывод

После выполнения тетради зафиксируйте в отчёте фактические значения MAE, RMSE и R² на тестовой выборке. Важно оценивать качество по test-части, которая не участвовала ни в обучении, ни в выборе конфигурации модели.